In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import duckdb

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

con = duckdb.connect()

training_data = con.read_parquet(
    str(
        PROCESSED_DIR
        / "historical_training_features_cutoff_20161130.parquet"
    )
).df()

evaluation_data = con.read_parquet(
    str(
        PROCESSED_DIR
        / "candidate_transaction_features_v2_cutoff_20170228.parquet"
    )
).df()

feature_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

X_train = training_data[feature_columns].astype("float64")
y_train = training_data["is_churn"].astype("int8")

X_eval = evaluation_data[feature_columns].astype("float64")
y_eval = evaluation_data["is_churn"].astype("int8")

assert training_data["msno"].is_unique
assert evaluation_data["msno"].is_unique

print("Training rows:", len(X_train))
print("Evaluation rows:", len(X_eval))

Training rows: 882127
Evaluation rows: 970960


In [2]:
models = {
    "Logistic regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=42),
        ),
    ]),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=100,
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    ),
    "LightGBM": LGBMClassifier(
        objective="binary",
        n_estimators=100,
        random_state=42,
        n_jobs=4,
    ),
}

## Create an Internal Historical Validation Split

In [3]:
from sklearn.model_selection import (
    train_test_split,
    PredefinedSplit,
    RandomizedSearchCV,
)

row_indices = np.arange(len(X_train))

inner_train_indices, inner_valid_indices = train_test_split(
    row_indices,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

test_fold = np.full(len(X_train), -1, dtype=int)
test_fold[inner_valid_indices] = 0

internal_split = PredefinedSplit(test_fold)

assert set(inner_train_indices).isdisjoint(inner_valid_indices)

print("Internal training rows:", len(inner_train_indices))
print("Internal validation rows:", len(inner_valid_indices))

print(
    "Internal training churn rate:",
    f"{y_train.iloc[inner_train_indices].mean():.2%}",
)

print(
    "Internal validation churn rate:",
    f"{y_train.iloc[inner_valid_indices].mean():.2%}",
)

Internal training rows: 705701
Internal validation rows: 176426
Internal training churn rate: 9.66%
Internal validation churn rate: 9.66%


## Define Search Spaces and Budgets

In [4]:
search_spaces = {
    "Logistic regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
    },
    "XGBoost": {
        "n_estimators": [100, 200, 400],
        "max_depth": [2, 3, 4, 6],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_weight": [1, 5, 20],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1.0, 5.0, 20.0],
    },
    "LightGBM": {
        "n_estimators": [100, 200, 400],
        "num_leaves": [7, 15, 31],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_samples": [20, 100, 500],
        "subsample": [0.8, 1.0],
        "subsample_freq": [1],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [0.0, 1.0, 10.0],
    },
}

search_budgets = {
    "Logistic regression": 4,
    "XGBoost": 8,
    "LightGBM": 8,
}

## Select Parameters Using Historical Data Only

In [5]:
searches = {}
tuning_rows = []

for name, model in models.items():
    print(f"Tuning {name}...", flush=True)

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=search_spaces[name],
        n_iter=search_budgets[name],
        scoring="neg_log_loss",
        cv=internal_split,
        refit=True,
        random_state=42,
        n_jobs=1,
        verbose=1,
        error_score="raise",
        return_train_score=True,
    )

    start = perf_counter()
    search.fit(X_train, y_train)
    search_seconds = perf_counter() - start

    searches[name] = search

    tuning_rows.append({
        "model": name,
        "best_internal_log_loss": -search.best_score_,
        "candidates_evaluated": len(search.cv_results_["params"]),
        "search_seconds": search_seconds,
        "best_parameters": search.best_params_,
    })

    print("Best parameters:", search.best_params_)
    print(f"Best internal Log Loss: {-search.best_score_:.6f}")

tuning_summary = (
    pd.DataFrame(tuning_rows)
    .sort_values("best_internal_log_loss")
    .reset_index(drop=True)
)

tuning_summary

Tuning Logistic regression...
Fitting 1 folds for each of 4 candidates, totalling 4 fits
Best parameters: {'classifier__C': 10.0}
Best internal Log Loss: 0.194477
Tuning XGBoost...
Fitting 1 folds for each of 8 candidates, totalling 8 fits
Best parameters: {'subsample': 0.8, 'reg_lambda': 20.0, 'n_estimators': 400, 'min_child_weight': 20, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Best internal Log Loss: 0.132340
Tuning LightGBM...
Fitting 1 folds for each of 8 candidates, totalling 8 fits
[LightGBM] [Info] Number of positive: 68190, number of negative: 637511
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010153 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 158
[LightGBM] [Info] Number of data points in the train set: 705701, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.096627 -> i

,model,best_internal_log_loss,candidates_evaluated,search_seconds,best_parameters
0,LightGBM,0.132262,8,49.050080,"{'subsample_freq': 1, 'subsample': 0.8, 'reg_l..."
1,XGBoost,0.132340,8,32.615914,"{'subsample': 0.8, 'reg_lambda': 20.0, 'n_esti..."
2,Logistic regression,0.194477,4,3.612655,{'classifier__C': 10.0}


In [6]:
import json

for name, search in searches.items():
    print(name)
    print(json.dumps(search.best_params_, indent=2))
    print()

Logistic regression
{
  "classifier__C": 10.0
}

XGBoost
{
  "subsample": 0.8,
  "reg_lambda": 20.0,
  "n_estimators": 400,
  "min_child_weight": 20,
  "max_depth": 6,
  "learning_rate": 0.1,
  "colsample_bytree": 1.0
}

LightGBM
{
  "subsample_freq": 1,
  "subsample": 0.8,
  "reg_lambda": 10.0,
  "num_leaves": 31,
  "n_estimators": 400,
  "min_child_samples": 100,
  "learning_rate": 0.03,
  "colsample_bytree": 0.8
}



## Evaluate Selected Models on the Later Cohort

In [7]:
tuned_rows = []

for name, search in searches.items():
    probabilities = search.best_estimator_.predict_proba(
        X_eval
    )[:, 1]

    assert np.isfinite(probabilities).all()
    assert (
        (probabilities >= 0)
        & (probabilities <= 1)
    ).all()

    tuned_rows.append({
        "model": name,
        "log_loss": log_loss(y_eval, probabilities),
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "average_precision": average_precision_score(
            y_eval, probabilities
        ),
    })

tuned_results = (
    pd.DataFrame(tuned_rows)
    .sort_values("log_loss")
    .reset_index(drop=True)
)

tuned_results

,model,log_loss,roc_auc,average_precision
0,Logistic regression,0.212925,0.829581,0.536523
1,LightGBM,0.234045,0.827786,0.482806
2,XGBoost,0.237186,0.825799,0.478728


In [8]:
untuned_results = pd.read_csv(
    REPORT_DIR / "untuned_model_comparison.csv"
)

tuning_comparison = untuned_results[
    ["model", "log_loss", "roc_auc", "average_precision"]
].merge(
    tuned_results,
    on="model",
    validate="one_to_one",
    suffixes=("_untuned", "_tuned"),
)

tuning_comparison["log_loss_improvement"] = (
    tuning_comparison["log_loss_untuned"]
    - tuning_comparison["log_loss_tuned"]
)

tuning_comparison["log_loss_improvement_pct"] = (
    100
    * tuning_comparison["log_loss_improvement"]
    / tuning_comparison["log_loss_untuned"]
)

tuning_comparison[
    [
        "model",
        "log_loss_untuned",
        "log_loss_tuned",
        "log_loss_improvement_pct",
        "roc_auc_tuned",
        "average_precision_tuned",
    ]
]

,model,log_loss_untuned,log_loss_tuned,log_loss_improvement_pct,roc_auc_tuned,average_precision_tuned
0,Logistic regression,0.212926,0.212925,0.000451,0.829581,0.536523
1,XGBoost,0.229378,0.237186,-3.403997,0.825799,0.478728
2,LightGBM,0.229887,0.234045,-1.808506,0.827786,0.482806


In [9]:
import json
import platform
import sklearn
import xgboost
import lightgbm

REPORT_DIR.mkdir(parents=True, exist_ok=True)

tuning_summary.to_csv(
    REPORT_DIR / "hyperparameter_tuning_summary.csv",
    index=False,
)

tuned_results.to_csv(
    REPORT_DIR / "tuned_model_results.csv",
    index=False,
)

tuning_comparison.to_csv(
    REPORT_DIR / "tuning_comparison.csv",
    index=False,
)

model_slugs = {
    "Logistic regression": "logistic_regression",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
}

for name, search in searches.items():
    pd.DataFrame(search.cv_results_).to_csv(
        REPORT_DIR / f"{model_slugs[name]}_search_results.csv",
        index=False,
    )

tuning_config = {
    "selection_method": "stratified internal historical holdout",
    "internal_validation_fraction": 0.2,
    "random_state": 42,
    "selection_metric": "log_loss",
    "training_feature_cutoff": "2016-11-30",
    "training_outcome_end": "2017-01-30",
    "later_evaluation_feature_cutoff": "2017-02-28",
    "refit_on_all_historical_data": True,
    "early_stopping_used": False,
    "later_cohort_used_in_parameter_search": False,
    "independent_test_evaluation_completed": False,
    "search_spaces": search_spaces,
    "search_budgets": search_budgets,
    "best_parameters": {
        name: search.best_params_
        for name, search in searches.items()
    },
    "versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
    },
}

(REPORT_DIR / "hyperparameter_tuning_config.json").write_text(
    json.dumps(tuning_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Search records, selected parameters, and evaluation results saved.")

Search records, selected parameters, and evaluation results saved.
